# PCF Estimation with Local LLM on Colab GPU

Runs the full Carbon Catalogue evaluation (866 products × 2 LLM methods) using
Llama 3.1 8B via Ollama on a Colab T4 GPU.

**Runtime**: ~30–60 minutes on T4.  
**Requirements**: Colab GPU runtime, Google Drive mounted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess, sys, shutil

DRIVE_BASE = Path('/content/drive/MyDrive')
REPO_URL = 'https://github.com/andersvestrum/carbon-aware-recsys.git'
REPO_BRANCH = 'big_man_thing'
REPO_ROOT = Path('/content/carbon-aware-recsys')
DRIVE_RESULTS = DRIVE_BASE / 'carbon-aware-recsys-colab' / 'results' / 'carbon'

LLM_MODEL = 'llama3.1:8b'

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
subprocess.run(
    ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')],
    check=True,
)
print('Repo cloned and dependencies installed.')

In [ ]:
%%bash
# Install Ollama
curl -fsSL https://ollama.com/install.sh | sh
echo "Ollama installed: $(ollama --version)"

In [ ]:
import subprocess, time, requests

proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print(f'Ollama server ready (took {i+1}s)')
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError('Ollama server failed to start')

In [ ]:
import subprocess
subprocess.run(['ollama', 'pull', LLM_MODEL], check=True)

In [ ]:
import time, requests, json

payload = {
    'model': LLM_MODEL,
    'temperature': 0,
    'max_tokens': 512,
    'messages': [
        {'role': 'system', 'content': 'Estimate PCF in kg CO2e. Output: PCF: X.X kg CO2e'},
        {'role': 'user', 'content': 'Product: Samsung 65-inch LED TV\nPCF: ?'},
    ],
}

t0 = time.time()
r = requests.post('http://localhost:11434/v1/chat/completions', json=payload, timeout=120)
elapsed = time.time() - t0

reply = r.json()['choices'][0]['message']['content']
print(f'Warm-up call: {elapsed:.1f}s')
print(f'Response (last 120 chars): ...{reply[-120:]}')
print(f'\nEstimated time for 1,732 calls: {1732 * elapsed / 60:.0f} minutes')

In [ ]:
import subprocess, sys, os

# Apply the two patches that haven't been pushed yet:
# 1. Remove the 50k metadata cap
# 2. Support --evaluation-limit -1 (= full catalogue)
# 3. Increase LLM timeout for safety

loader_path = REPO_ROOT / 'src' / 'data' / 'amazon_loader.py'
text = loader_path.read_text()
text = text.replace('MAX_META_ROWS = 50_000  # TODO: remove cap for full runs', 'MAX_META_ROWS = None')
loader_path.write_text(text)

retrieval_path = REPO_ROOT / 'src' / 'carbon' / 'retrieval.py'
text = retrieval_path.read_text()
text = text.replace('timeout: float = 60.0,', 'timeout: float = 300.0,')
retrieval_path.write_text(text)

script_path = REPO_ROOT / 'scripts' / 'predict_carbon.py'
text = script_path.read_text()
old_fn = '''def _resolve_evaluation_limit(
    requested_limit: int | None,
    llm_client: OpenAILLMClient | None,
) -> int | None:
    if requested_limit is not None or llm_client is None:
        return requested_limit

    log.info(
        "LLM baselines enabled without --evaluation-limit. "
        "Defaulting to 100 evaluation examples to control cost."
    )
    return 100'''
new_fn = '''def _resolve_evaluation_limit(
    requested_limit: int | None,
    llm_client: OpenAILLMClient | None,
) -> int | None:
    if requested_limit is not None:
        return None if requested_limit < 0 else requested_limit
    if llm_client is None:
        return None

    log.info(
        "LLM baselines enabled without --evaluation-limit. "
        "Defaulting to 100 evaluation examples to control cost. "
        "Pass --evaluation-limit -1 to evaluate the full catalogue."
    )
    return 100'''
text = text.replace(old_fn, new_fn)
script_path.write_text(text)

print('Patches applied.')

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT))

from src.data.carbon_loader import load_carbon_catalogue

df = load_carbon_catalogue()
print(f'\nCarbon Catalogue ready: {len(df)} products, {len(df.columns)} columns')

In [ ]:
import subprocess, sys, os, time

env = os.environ.copy()
env['OPENAI_API_KEY'] = 'unused'
env['OPENAI_BASE_URL'] = 'http://localhost:11434/v1'

cmd = [
    sys.executable,
    str(REPO_ROOT / 'scripts' / 'predict_carbon.py'),
    '--device', 'cpu',
    '--num-threads', '4',
    '--llm-model', LLM_MODEL,
    '--evaluation-limit', '-1',
    '--amazon-limit', '0',
]

print('Running:', ' '.join(cmd))
t0 = time.time()

result = subprocess.run(cmd, cwd=str(REPO_ROOT), env=env)

elapsed_min = (time.time() - t0) / 60
print(f'\nFinished in {elapsed_min:.1f} minutes (exit code {result.returncode})')

In [ ]:
import pandas as pd

metrics = pd.read_csv(REPO_ROOT / 'output' / 'results' / 'carbon' / 'pcf_evaluation_metrics.csv')
print('=== Evaluation Metrics ===')
print(metrics.to_string(index=False))

meta = pd.read_json(REPO_ROOT / 'output' / 'results' / 'carbon' / 'pcf_run_metadata.json', typ='series')
print('\n=== Run Metadata ===')
print(meta.to_string())

In [ ]:
import shutil

DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

src_dir = REPO_ROOT / 'output' / 'results' / 'carbon'
for f in src_dir.iterdir():
    shutil.copy2(f, DRIVE_RESULTS / f.name)
    print(f'Copied {f.name} -> {DRIVE_RESULTS / f.name}')

cache_src = REPO_ROOT / 'data' / 'processed' / 'carbon' / 'llm_prediction_cache.jsonl'
if cache_src.exists():
    cache_dst = DRIVE_RESULTS / 'llm_prediction_cache.jsonl'
    shutil.copy2(cache_src, cache_dst)
    print(f'Copied LLM cache -> {cache_dst}')

print('\nAll results saved to Google Drive.')